# Cascade netem validation (steps 2-6)

Assumes step 1 is done: `bob_cascade_driver.py` has `arm_cascade_netem` moved above the `try/except RuntimeError` block, and `df.upload_project(slice_obj)` has been run to push that fix to all nodes.

Order matters -- each step assumes the previous one passed:

2. Real-interface arm/disarm mechanics (not `lo`)
3. One real end-to-end pair through both driver scripts (baseline vs one delay value)
4. Isolation check on that pair (reconciliation time moves, rest of pipeline doesn't)
5. Forced non-convergence with delay active, on real nodes
6. Negative test: bogus `--cascade-iface` should crash the whole process, not produce `non_convergent: true`

Stop and fix, don't skip ahead, if any step fails.

In [12]:
df.upload_project(slice_obj)


=== Uploading project (clean tarball) ===
  Uploading to alice...
  Uploading to bob...
  Uploading to switch...
  Upload complete (qne + validation + scenarios + p4 on every node)


In [1]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as df
fablib = df.get_fablib()

SLICE_NAME = 'qfabric-bb84-2'  # from 01_setup_slice.ipynb -- reconnect, don't reprovision
slice_obj = fablib.get_slice(name=SLICE_NAME)
print(f"Reconnected to slice '{SLICE_NAME}' (state: {slice_obj.get_state()})")

alice_node = slice_obj.get_node("alice")
bob_node = slice_obj.get_node("bob")

# STOP if this fix hasn't been applied and synced yet -- everything below
# assumes it has.
print("Before proceeding: confirm bob_cascade_driver.py has the arm-outside-try")
print("fix applied AND df.upload_project(slice_obj) has been re-run since.")

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid
Reconnected to slice 'qfabric-bb84-2' (state: StableOK)
Before proceeding: confirm bob_cascade_driver.py has the arm-outside-try
fix applied AND df.upload_project(slice_obj) has been re-run since.


In [2]:
import json
import time

ALICE_IFACE = alice_node.get_interface(network_name="net_alice_switch").get_device_name()
BOB_IFACE = bob_node.get_interface(network_name="net_switch_bob").get_device_name()
print(f"ALICE_IFACE = {ALICE_IFACE}")
print(f"BOB_IFACE = {BOB_IFACE}")

PORT = 5200
VENV_PY = "~/qfabric/.venv/bin/python3"
REMOTE_DIR = "~/qfabric"

def tc_check(node, iface, label, delay_ms=20):
    """Direct arm/disarm check against a real node + real interface --
    same functions the driver scripts call, no protocol traffic involved.
    Returns (arm_dict, disarm_dict) or raises if either step's JSON is
    missing/malformed (fail loud, don't silently treat a crash as a skip).
    """
    cmd = (
        f"cd {REMOTE_DIR} && {VENV_PY} -c \""
        f"from qne.netem import arm_cascade_netem, disarm_cascade_netem; "
        f"import json; "
        f"print(json.dumps(arm_cascade_netem('{iface}', {PORT}, delay_ms={delay_ms}))); "
        f"print(json.dumps(disarm_cascade_netem('{iface}')))"
        f"\""
    )
    stdout, stderr = node.execute(cmd, quiet=True)
    print(f"--- [{label}] stderr ---")
    print(stderr or "(empty)")
    lines = [l for l in stdout.strip().split("\n") if l.strip().startswith("{")]
    assert len(lines) == 2, f"[{label}] expected 2 JSON lines (arm, disarm), got: {stdout}"
    arm, disarm = json.loads(lines[0]), json.loads(lines[1])
    assert "netem" in arm["verify_qdisc_show"], f"[{label}] netem not present after arm: {arm}"
    assert "netem" not in disarm["verify_qdisc_show"], f"[{label}] netem still present after disarm: {disarm}"
    print(f"[{label}] PASS -- armed: {arm['verify_qdisc_show'].strip()!r}")
    print(f"[{label}] PASS -- disarmed: {disarm['verify_qdisc_show'].strip()!r}")
    return arm, disarm

print("=== STEP 2: real-interface arm/disarm mechanics ===")
_ = tc_check(alice_node, ALICE_IFACE, "alice_real_iface")
_ = tc_check(bob_node, BOB_IFACE, "bob_real_iface")
print("STEP 2 PASS")

ALICE_IFACE = enp7s0
BOB_IFACE = enp7s0
=== STEP 2: real-interface arm/disarm mechanics ===
--- [alice_real_iface] stderr ---
(empty)
[alice_real_iface] PASS -- armed: 'qdisc prio 1: root refcnt 33 bands 3 priomap 1 2 2 2 1 2 0 0 1 1 1 1 1 1 1 1\nqdisc netem 30: parent 1:3 limit 1000 delay 20ms'
[alice_real_iface] PASS -- disarmed: 'qdisc mq 0: root \nqdisc fq_codel 0: parent :4 limit 10240p flows 1024 quantum 1514 target 5ms interval 100ms memory_limit 32Mb ecn drop_batch 64 \nqdisc fq_codel 0: parent :3 limit 10240p flows 1024 quantum 1514 target 5ms interval 100ms memory_limit 32Mb ecn drop_batch 64 \nqdisc fq_codel 0: parent :2 limit 10240p flows 1024 quantum 1514 target 5ms interval 100ms memory_limit 32Mb ecn drop_batch 64 \nqdisc fq_codel 0: parent :1 limit 10240p flows 1024 quantum 1514 target 5ms interval 100ms memory_limit 32Mb ecn drop_batch 64'
--- [bob_real_iface] stderr ---
(empty)
[bob_real_iface] PASS -- armed: 'qdisc prio 1: root refcnt 33 bands 3 priomap 1 2 2 2 1 2 0

## Steps 3-4: one real end-to-end pair, baseline vs delay, isolation check

In [7]:
# Sync current code (including the step-1 fix) to all nodes before running anything real.
df.upload_project(slice_obj)

REMOTE_BOB_KEY_JSON = f"{REMOTE_DIR}/results/bob_sifted_bits_key0.json"
REMOTE_ALICE_KEY_JSON = f"{REMOTE_DIR}/results/alice_sifted_bits_key0.json"
REMOTE_QBER = 0.01282051282051282   # key0, from key_pairs_metadata.csv
REMOTE_K = 390                       # key0, from key_pairs_metadata.csv

def run_fabric_pair(delay_ms, label, extra_bob_flags=""):
    """Launch Bob (background thread, execute_thread) then Alice (blocking),
    following the same pattern deploy_fabric.py already uses elsewhere.
    extra_bob_flags lets step 5/6 add --reconciliation-prob / a bogus iface
    without duplicating this function.
    """
    cascade_flags = f"--cascade-iface {{iface}} --cascade-delay-ms {delay_ms}" if delay_ms else ""

    alice_node.execute("sudo pkill -9 -f alice_cascade_responder 2>/dev/null; sleep 1", quiet=True)
    bob_node.execute("sudo pkill -9 -f bob_cascade_driver 2>/dev/null; sleep 1", quiet=True)

    bob_out = f"results/test_bob_{label}.json"
    bob_cascade_flags = cascade_flags.format(iface=BOB_IFACE)
    bob_thread = bob_node.execute_thread(
        f"cd {REMOTE_DIR} && {VENV_PY} scripts/bob_cascade_driver.py "
        f"--key-json {REMOTE_BOB_KEY_JSON} --alice-key-json {REMOTE_ALICE_KEY_JSON} "
        f"--host 0.0.0.0 --port {PORT} --qber {REMOTE_QBER} --k {REMOTE_K} "
        f"--output {bob_out} --no-unique-suffix {bob_cascade_flags} {extra_bob_flags} "
        f"2>&1"
    )
    time.sleep(5)

    bob_classical_ip = bob_node.get_interface(network_name="net_switch_bob").get_ip_addr()
    alice_out = f"results/test_alice_{label}.json"
    alice_cascade_flags = cascade_flags.format(iface=ALICE_IFACE)
    alice_thread = alice_node.execute_thread(
        f"cd {REMOTE_DIR} && {VENV_PY} scripts/alice_cascade_responder.py "
        f"--key-json {REMOTE_ALICE_KEY_JSON} --bob-host {bob_classical_ip} --port {PORT} "
        f"--output {alice_out} --no-unique-suffix {alice_cascade_flags} "
        f"2>&1"
    )

    alice_stdout, _ = alice_thread.result(timeout=180)
    bob_stdout, _ = bob_thread.result(timeout=180)
    print(f"--- [{label}] bob tail ---\n{bob_stdout[-1000:]}")
    print(f"--- [{label}] alice tail ---\n{alice_stdout[-500:]}")

    bob_json, _ = bob_node.execute(f"cat {REMOTE_DIR}/{bob_out} 2>/dev/null || echo '{{}}'", quiet=True)
    return json.loads(bob_json) if bob_json.strip().startswith("{") else None

print("=== STEP 3: baseline (0ms) ===")
r_baseline = run_fabric_pair(0, "step3_baseline")
assert r_baseline is not None, "baseline run produced no output -- fix before testing delay"
print(f"non_convergent={r_baseline['non_convergent']}  reconciliation_elapsed_seconds={r_baseline['reconciliation_elapsed_seconds']:.3f}")

print("\n=== STEP 3: delay (50ms) ===")
r_delay = run_fabric_pair(50, "step3_delay50")
assert r_delay is not None, "delay run produced no output -- fix before proceeding"
print(f"non_convergent={r_delay['non_convergent']}  reconciliation_elapsed_seconds={r_delay['reconciliation_elapsed_seconds']:.3f}")
print("cascade_netem:", json.dumps(r_delay["cascade_netem"], indent=2))


=== Uploading project (clean tarball) ===
  Uploading to alice...
  Uploading to bob...
  Uploading to switch...
  Upload complete (qne + validation + scenarios + p4 on every node)
=== STEP 3: baseline (0ms) ===
--- [step3_baseline] bob tail ---
Bob: waiting for Alice on 0.0.0.0:5200...
Bob: Alice connected, starting reconciliation
Bob: done. Result written to results/test_bob_step3_baseline.json
{
  "toeplitz_prob": 0.0,
  "final_key_prob": 0.0,
  "reconciliation_prob": 0.0,
  "verify_digest_prob": 0.0,
  "seed": 42,
  "output_path": "results/test_bob_step3_baseline.json",
  "k_pe": 390,
  "total_corrections": 33,
  "secure_key_length": 1714,
  "t_verify": 34,
  "digest_length": 34,
  "verification_passed": true,
  "remaining_errors_after_reconciliation": 0,
  "non_convergent": false,
  "error": null,
  "elapsed_seconds": 7.187350273132324,
  "faults_fired": {},
  "length_mode": "placeholder",
  "cascade_netem": {},
  "reconciliation_elapsed_seconds": 0.2372729778289795
}

--- [step3

In [8]:
print("=== STEP 4: isolation check ===")
recon_baseline = r_baseline["reconciliation_elapsed_seconds"]
recon_delay = r_delay["reconciliation_elapsed_seconds"]
rest_baseline = r_baseline["elapsed_seconds"] - recon_baseline
rest_delay = r_delay["elapsed_seconds"] - recon_delay

print(f"reconciliation_elapsed_seconds: baseline={recon_baseline:.3f}s  delay50ms={recon_delay:.3f}s  (delta={recon_delay-recon_baseline:+.3f}s)")
print(f"rest_of_pipeline (PA+verify):    baseline={rest_baseline:.3f}s  delay50ms={rest_delay:.3f}s  (delta={rest_delay-rest_baseline:+.3f}s)")

if recon_delay > recon_baseline and abs(rest_delay - rest_baseline) < 0.5 * rest_baseline + 0.1:
    print("STEP 4 PASS -- reconciliation moved, rest of pipeline stayed roughly flat")
else:
    print("STEP 4 NEEDS A LOOK -- either reconciliation didn't move, or the rest of the "
          "pipeline moved too (impairment may be leaking outside the Cascade phase). "
          "Don't proceed to a real sweep until this is understood.")

=== STEP 4: isolation check ===
reconciliation_elapsed_seconds: baseline=0.237s  delay50ms=1.942s  (delta=+1.704s)
rest_of_pipeline (PA+verify):    baseline=6.950s  delay50ms=5.364s  (delta=-1.586s)
STEP 4 PASS -- reconciliation moved, rest of pipeline stayed roughly flat


## Step 5: forced non-convergence with delay active

In [16]:
print("=== STEP 5: forced non-convergence, delay active (deterministic) ===")
r_forced = run_fabric_pair(50, "step5_forced_fail_v3", extra_bob_flags="--force-non-convergent")
assert r_forced is not None, "run produced no output at all -- check stderr above"
print(f"non_convergent={r_forced['non_convergent']}")
assert r_forced["non_convergent"] is True, "still didn't raise -- check the monkeypatch actually applied"

netem5 = r_forced["cascade_netem"]
assert "disarm" in netem5, f"disarm missing -- fix did not take effect: {netem5}"
print("disarm present:", netem5["disarm"])

alice_check, _ = alice_node.execute(f"sudo tc qdisc show dev {ALICE_IFACE}", quiet=True)
bob_check, _ = bob_node.execute(f"sudo tc qdisc show dev {BOB_IFACE}", quiet=True)
assert "netem" not in alice_check and "netem" not in bob_check, \
    "netem left armed on a real interface after a forced failure -- LEAK"
print("STEP 5 PASS (genuinely -- RuntimeError path actually exercised)")

=== STEP 5: forced non-convergence, delay active (deterministic) ===
--- [step5_forced_fail_v3] bob tail ---
      "armed_at": 1788984100.594836,
      "iface": "enp7s0",
      "port": 5200,
      "netem_spec": "netem delay 50.0ms",
      "verify_qdisc_show": "qdisc prio 1: root refcnt 33 bands 3 priomap 1 2 2 2 1 2 0 0 1 1 1 1 1 1 1 1\nqdisc netem 30: parent 1:3 limit 1000 delay 50ms"
    },
    "disarm": {
      "disarmed_at": 1788984100.7291052,
      "iface": "enp7s0",
      "verify_qdisc_show": "qdisc mq 0: root \nqdisc fq_codel 0: parent :4 limit 10240p flows 1024 quantum 1514 target 5ms interval 100ms memory_limit 32Mb ecn drop_batch 64 \nqdisc fq_codel 0: parent :3 limit 10240p flows 1024 quantum 1514 target 5ms interval 100ms memory_limit 32Mb ecn drop_batch 64 \nqdisc fq_codel 0: parent :2 limit 10240p flows 1024 quantum 1514 target 5ms interval 100ms memory_limit 32Mb ecn drop_batch 64 \nqdisc fq_codel 0: parent :1 limit 10240p flows 1024 quantum 1514 target 5ms interval 100

In [15]:
df.upload_project(slice_obj)

# 3. Confirm it landed on the actual node this time
stdout, _ = bob_node.execute("grep -n 'force-non-convergent' ~/qfabric/scripts/bob_cascade_driver.py", quiet=True)
print(stdout or "STILL NOT FOUND on bob_node -- upload didn't take")


=== Uploading project (clean tarball) ===
  Uploading to alice...
  Uploading to bob...
  Uploading to switch...
  Upload complete (qne + validation + scenarios + p4 on every node)
82:    parser.add_argument("--force-non-convergent", action="store_true",



## Step 6: negative test -- bogus iface should crash loudly, not produce a misleading result

In [18]:
print("=== STEP 6: bogus --cascade-iface, expect a crash (no output JSON) ===")
bob_node.execute("sudo pkill -9 -f bob_cascade_driver 2>/dev/null; sleep 1", quiet=True)
alice_node.execute("sudo pkill -9 -f alice_cascade_responder 2>/dev/null; sleep 1", quiet=True)

bad_out = "results/test_bob_step6_bogus_iface.json"
bob_node.execute(f"rm -f {REMOTE_DIR}/{bad_out}", quiet=True)

bad_cmd = (
    f"cd {REMOTE_DIR} && {VENV_PY} scripts/bob_cascade_driver.py "
    f"--key-json {REMOTE_BOB_KEY_JSON} --alice-key-json {REMOTE_ALICE_KEY_JSON} "
    f"--host 0.0.0.0 --port {PORT} --qber {REMOTE_QBER} --k {REMOTE_K} "
    f"--output {bad_out} --no-unique-suffix "
    f"--cascade-iface totally_bogus_iface_xyz --cascade-delay-ms 20 2>&1"
)
bob_thread = bob_node.execute_thread(bad_cmd)
time.sleep(5)  # let Bob bind and reach accept()

# A normal Alice, so Bob's accept() unblocks and Bob actually reaches the
# (bogus-iface) arm call -- without this, Bob just hangs at accept() forever.
alice_cmd = (
    f"cd {REMOTE_DIR} && {VENV_PY} scripts/alice_cascade_responder.py "
    f"--key-json {REMOTE_ALICE_KEY_JSON} --bob-host {bob_node.get_interface(network_name='net_switch_bob').get_ip_addr()} "
    f"--port {PORT} --output results/test_alice_step6.json --no-unique-suffix 2>&1"
)
alice_thread = alice_node.execute_thread(alice_cmd)

stdout, _ = bob_thread.result(timeout=30)
alice_stdout, _ = alice_thread.result(timeout=30)
print("--- bob stdout/stderr ---")
print(stdout[-1500:])
print("--- alice stdout/stderr (expected to fail/drop -- that's fine here) ---")
print(alice_stdout[-500:])

exists, _ = bob_node.execute(f"test -f {REMOTE_DIR}/{bad_out} && echo YES || echo NO", quiet=True)
print(f"output file exists: {exists.strip()}")
assert exists.strip() == "NO", "output JSON produced despite bogus iface -- fix didn't fully take"
assert "arm_cascade_netem failed" in stdout, "failed for a different reason than expected -- read traceback"
print("STEP 6 PASS")

=== STEP 6: bogus --cascade-iface, expect a crash (no output JSON) ===
--- bob stdout/stderr ---
Bob: waiting for Alice on 0.0.0.0:5200...
Bob: Alice connected, starting reconciliation
Traceback (most recent call last):
  File "/home/ubuntu/qfabric/scripts/bob_cascade_driver.py", line 143, in <module>
    netem_info["arm"] = arm_cascade_netem(
  File "/home/ubuntu/qfabric/qne/netem.py", line 59, in arm_cascade_netem
    raise RuntimeError(f"arm_cascade_netem failed on {iface}:{port} (rc={rc}): {out}")
RuntimeError: arm_cascade_netem failed on totally_bogus_iface_xyz:5200 (rc=1): Cannot find device "totally_bogus_i"


--- alice stdout/stderr (expected to fail/drop -- that's fine here) ---
Alice: loaded key with 3504 bits
Alice: connecting to Bob at 10.10.1.2:5200...
Alice: channel closed during Cascade phase: Connection closed while receiving data

output file exists: NO
STEP 6 PASS


## All steps passed?
If steps 2-6 above all printed PASS: the harness is validated end-to-end on real hardware, both for the happy path and for the two failure modes that matter (Cascade non-convergence, and netem infrastructure failure) staying correctly distinguishable. Real sweep data collection can start -- in a new notebook, reusing `sweep_utils.py` patterns, not this validation harness.